# Course 3: Large Language Models
## Session 3: Pretraining data and scaling laws

**Goal:** train three tiny models, fit a small scaling law, and use it to make and test a prediction.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every cell without a NOT RUN label was run once on a CPU (November 2026) with tiny models and a small synthetic Bangla corpus. Cells labelled NOT RUN need a GPU, a model download or an account; run them on Kaggle or Colab.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

@dataclass
class Config:
    vocab_size: int = 8000
    block_size: int = 256
    n_layer: int = 6
    n_head: int = 4
    d_model: int = 256
    dropout: float = 0.0

class Block(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.n_head = c.n_head
        self.ln1 = nn.LayerNorm(c.d_model)
        self.qkv = nn.Linear(c.d_model, 3 * c.d_model)
        self.proj = nn.Linear(c.d_model, c.d_model)
        self.ln2 = nn.LayerNorm(c.d_model)
        self.fc = nn.Linear(c.d_model, 4 * c.d_model)
        self.out = nn.Linear(4 * c.d_model, c.d_model)
        self.drop = nn.Dropout(c.dropout)

    def attn(self, x):
        B, T, D = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        split = lambda t: t.view(B, T, self.n_head, D // self.n_head).transpose(1, 2)
        y = F.scaled_dot_product_attention(split(q), split(k), split(v), is_causal=True)
        return self.proj(y.transpose(1, 2).reshape(B, T, D))

    def forward(self, x):
        x = x + self.drop(self.attn(self.ln1(x)))
        x = x + self.drop(self.out(F.gelu(self.fc(self.ln2(x)))))
        return x

class GPT(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.c = c
        self.tok = nn.Embedding(c.vocab_size, c.d_model)
        self.pos = nn.Embedding(c.block_size, c.d_model)
        self.blocks = nn.ModuleList([Block(c) for _ in range(c.n_layer)])
        self.ln = nn.LayerNorm(c.d_model)
        self.head = nn.Linear(c.d_model, c.vocab_size, bias=False)
        self.head.weight = self.tok.weight          # weight tying
        self.apply(self._init)

    def _init(self, m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, std=0.02)
        if isinstance(m, nn.Linear) and m.bias is not None:
            nn.init.zeros_(m.bias)

    def forward(self, idx, targets=None, return_hidden=False):
        B, T = idx.shape
        x = self.tok(idx) + self.pos(torch.arange(T, device=idx.device))
        for block in self.blocks:
            x = block(x)
        h = self.ln(x)
        if return_hidden:
            return h
        logits = self.head(h)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)),
                                   targets.view(-1), ignore_index=-100)
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_p=0.9):
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -self.c.block_size:])
            probs = torch.softmax(logits[:, -1, :] / temperature, dim=-1)
            sp, si = torch.sort(probs, descending=True)
            keep = torch.cumsum(sp, dim=-1) - sp < top_p
            sp = sp * keep
            sp = sp / sp.sum(dim=-1, keepdim=True)
            nxt = si.gather(-1, torch.multinomial(sp, 1))
            idx = torch.cat([idx, nxt], dim=1)
        return idx

In [ ]:
import random, itertools
SUBJ = ["মা", "বাবা", "শিক্ষক", "ছেলেটি", "মেয়েটি", "কৃষক", "দাদু", "রহিম"]
TIME = ["সকালে", "বিকেলে", "রাতে", "প্রতিদিন", "আজ"]
PLACE = ["বাজারে", "স্কুলে", "মাঠে", "বাড়িতে", "গ্রামে"]
OBJ_VERB = [("ভাত", "খায়"), ("বই", "পড়ে"), ("গান", "গায়"), ("ফুটবল", "খেলে"), ("চিঠি", "লেখে"), ("মাছ", "কেনে"), ("গল্প", "বলে"), ("ছবি", "আঁকে")]
VERB_OF = dict(OBJ_VERB)
# A toy language: subject, time, place, object, verb, full stop. The verb must fit the object (ভাত খায়, বই পড়ে).
ALL_SENTENCES = [[s, t, p, o, v, "।"] for s, t, p, (o, v) in itertools.product(SUBJ, TIME, PLACE, OBJ_VERB)]

def is_valid(words):
    return (len(words) == 6 and words[0] in SUBJ and words[1] in TIME and words[2] in PLACE
            and VERB_OF.get(words[3]) == words[4] and words[5] == "।")

EXTRA = ["<pad>", "<mask>", "প্রশ্ন", "উত্তর", ":", "?", "কী", "করে"]
VOCAB = EXTRA + SUBJ + TIME + PLACE + [o for o, _ in OBJ_VERB] + [v for _, v in OBJ_VERB] + ["।"]
STOI = {w: i for i, w in enumerate(VOCAB)}
enc = lambda words: [STOI[w] for w in words]
dec = lambda ids: [VOCAB[i] for i in ids]

In [ ]:
random.seed(0); torch.manual_seed(0)
_shuffled = ALL_SENTENCES[:]; random.shuffle(_shuffled)
train_sents, val_sents = _shuffled[:1200], _shuffled[1200:]            # 400 sentences the model never sees
train_ids = torch.tensor([i for s in train_sents for i in enc(s)])
val_ids = torch.tensor([i for s in val_sents for i in enc(s)])

def get_batch(data, block_size, batch_size):
    ix = torch.randint(0, (len(data) - block_size - 1) // 6, (batch_size,)) * 6      # start on a sentence boundary
    x = torch.stack([data[i:i + block_size] for i in ix]); y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
    return x, y

def train_lm(cfg, steps=600, lr=3e-3, data=None, seed=0):
    torch.manual_seed(seed); data = train_ids if data is None else data
    model = GPT(cfg); opt = torch.optim.AdamW(model.parameters(), lr=lr)
    for _ in range(steps):
        x, y = get_batch(data, cfg.block_size, 32)
        _, loss = model(x, y); opt.zero_grad(); loss.backward(); opt.step()
    return model

@torch.no_grad()
def val_loss(model, block_size=24):
    n = (len(val_ids) - 1) // block_size * block_size
    x = val_ids[:n].view(-1, block_size); y = val_ids[1:n + 1].view(-1, block_size)
    return model(x, y)[1].item()

@torch.no_grad()
def sample_sentences(model, n=200, temperature=1.0):
    prompt = torch.tensor([enc(train_sents[i]) for i in torch.randint(len(train_sents), (n,)).tolist()])   # one real sentence as the prompt
    out = model.generate(prompt, 6, temperature=temperature, top_p=1.0)[:, 6:]                     # the model writes the next one
    return [dec(row.tolist()) for row in out]

### 1. Three models, same data, same number of steps

In [ ]:
import math, numpy as np
floor = math.log(len(ALL_SENTENCES)) / 6                 # the best loss any model can reach on this language
results = []
for d in [4, 8, 16]:
    losses = []
    for seed in range(3):
        m = train_lm(Config(vocab_size=len(VOCAB), block_size=24, n_layer=2, n_head=2, d_model=d), steps=400, seed=seed)
        losses.append(val_loss(m))
    n_params = sum(p.numel() for p in m.parameters())
    results.append((n_params, np.mean(losses), np.std(losses)))
    print(f"d_model {d:3d}   parameters {n_params:6d}   validation loss {np.mean(losses):.3f} ± {np.std(losses):.3f}   above the floor by {np.mean(losses) - floor:.3f}")
assert results[0][1] > results[1][1] > results[2][1]

### 2. Fit a power law
Scaling-law papers fit `loss = floor + c * N^(-alpha)`. On a log-log plot of (loss - floor) against parameters N, that is a straight line.

In [ ]:
import matplotlib.pyplot as plt
N = np.array([r[0] for r in results]); excess = np.array([r[1] for r in results]) - floor
slope, intercept = np.polyfit(np.log(N), np.log(excess), 1)
print(f"fitted: loss = {floor:.3f} + {math.exp(intercept):.1f} * N^({slope:.2f})")
plt.loglog(N, excess, "o"); plt.loglog(N, np.exp(intercept) * N ** slope, "--"); plt.xlabel("parameters"); plt.ylabel("loss above the floor"); plt.savefig("scaling.png")

### 3. Test the prediction
A fit through three points is easy. The test of a scaling law is whether it predicts a model you have not trained yet.

In [ ]:
d_new = 32
m = train_lm(Config(vocab_size=len(VOCAB), block_size=24, n_layer=2, n_head=2, d_model=d_new), steps=400)
n_new = sum(p.numel() for p in m.parameters()); predicted = floor + math.exp(intercept) * n_new ** slope; actual = val_loss(m)
print(f"d_model {d_new}: parameters {n_new} | predicted loss {predicted:.3f} | actual loss {actual:.3f} | floor {floor:.3f}")

If the prediction is off, that is a finding, not a failure. Three small models, a fixed number of steps and a toy language are far from the conditions in which scaling laws were measured. Say so when you report it.

### 4. Compute-optimal training: the Chinchilla rule of thumb
Hoffmann et al. (2022) found that for a fixed compute budget, parameters and training tokens should grow together, at roughly 20 tokens per parameter. Training compute is about `6 * N * D` floating-point operations.

In [ ]:
def chinchilla(flops, tokens_per_param=20):
    n = math.sqrt(flops / (6 * tokens_per_param)); return n, tokens_per_param * n
for name, flops in [("one free GPU session (about 1e17)", 1e17), ("a small lab's month (about 1e20)", 1e20)]:
    n, dset = chinchilla(flops)
    print(f"{name:36s} about {n / 1e6:7.1f} million parameters trained on {dset / 1e9:6.2f} billion tokens")

For Bangla, the binding limit is often the data, not the compute: if you have 500 million clean tokens, the rule points to a model of about 25 million parameters. TinyStories (Eldan and Li 2023) showed that models that small can write fluent text when the data is simple and clean.

### 5. Data filtering in one cell
FineWeb-style pipelines drop duplicates and low-quality lines before training. Two cheap filters:

In [ ]:
raw = [" ".join(s) for s in train_sents[:50]] + [" ".join(train_sents[0])] * 10 + ["click here click here click here", "aaaa bbbb", "১২৩ ৪৫৬"]
seen, kept = set(), []
for line in raw:
    words = line.split()
    bangla_share = sum("\u0980" <= ch <= "\u09FF" for ch in line) / max(len(line.replace(" ", "")), 1)
    if line in seen or bangla_share < 0.8 or len(set(words)) < 0.6 * len(words) or len(words) < 4: continue
    seen.add(line); kept.append(line)
print(f"kept {len(kept)} of {len(raw)} lines")
assert len(kept) == 50

### TODO
1. Fix the model at d_model 32 and vary the data: 100, 300 and 1,200 training sentences. Fit loss against data size.
2. Repeat section 1 with 3 different step counts. Does the exponent change?
3. Write down the compute you have for the capstone and use `chinchilla` to choose a model size. Then check whether you have enough Bangla tokens for it.